# 10. Demo cases for the app test (M4-T06)

Puts small, ready-to-upload test scans in the shared Drive folder `BrainMRI_Data/demo_cases/`, so testers do not have to
download the large archives:

- `RHUH-0002/visit_0`, `visit_1`, `visit_2`: the three scans of RHUH-0002 (pre-op, early post-op, recurrence), cleaned
  with the team's cleaning step (`bmts.data.cleaning.standardize`), as used for M3-T02. Files are named
  `RHUH-0002_<visit>_<type>.nii.gz`, so the web interface recognises the MRI type and the scan.
- `BraTS2021_00110` (already there): the cleaned scans of validation patient BraTS2021_00110, as used for M3-T01.

Only train/validation patients go here, never locked test patients. A CPU runtime is enough; it takes about 5 minutes.
Run all cells (**Runtime -> Run all**).

In [ ]:
# 1. Drive and the team code
import os, sys, tarfile, hashlib
from pathlib import Path
from types import SimpleNamespace
from google.colab import drive
drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/BrainMRI_Data")
assert BASE.is_dir(), "BrainMRI_Data is not in My Drive: add a shortcut to it in My Drive, then run again"
if not Path("/content/repo/.git").exists():
    !git clone -q https://github.com/nhahub/NHA-5-187.git /content/repo
!git -C /content/repo pull -q && git -C /content/repo log --oneline -1
sys.path.insert(0, "/content/repo/src")

In [ ]:
# 2. Settings: one train/validation patient with several visits
import pandas as pd
from bmts.data.splits import check_subject
DATASET, SUBJECT = "rhuh", "RHUH-0002"
MODALITIES = ("t1", "t1ce", "t2", "flair")
TAR = BASE / "raw/longitudinal_rhuh_gbm/longitudinal_rhuh_gbm.tar"
OUT = BASE / "demo_cases" / SUBJECT
print(SUBJECT, "is in the", check_subject(BASE / "qc/split_v1.csv", DATASET, SUBJECT), "split")   # raises for locked patients
manifest = pd.read_csv("/content/repo/data/manifests/rhuh_manifest.csv", dtype=str)
rows = manifest[(manifest.subject_id == SUBJECT) & manifest.scan.isin(MODALITIES)]
print(len(rows), "files to extract:", sorted(rows.timepoint_raw.unique()), "visits")

In [ ]:
# 3. Extract only this patient's files from the archive (reads the archive once, about 2-3 minutes)
RAW = Path("/content/raw")
wanted = set(rows.path_in_archive)
with tarfile.open(TAR, "r|", bufsize=16 * 1024 * 1024) as tar:
    for member in tar:
        if member.name in wanted:
            tar.extract(member, RAW, filter="data")
            wanted.discard(member.name)
            if not wanted:
                break
assert not wanted, f"not found in the archive: {sorted(wanted)}"
for r in rows.itertuples():                     # same files as in the team's checksum list
    digest = hashlib.sha256((RAW / r.path_in_archive).read_bytes()).hexdigest()
    assert digest == r.sha256, f"checksum mismatch: {r.file_name}"
print("all", len(rows), "files extracted and checksums match")

In [ ]:
# 4. Clean them like the rest of the project (RAS orientation, 240x240x155, brain z-score) into demo_cases
from bmts.data.cleaning import standardize as S
for r in rows.itertuples():
    out = OUT / f"visit_{r.timepoint_raw}" / f"{SUBJECT}_{r.timepoint_raw}_{r.scan}.nii.gz"
    status, error = S.standardize_file(SimpleNamespace(dataset=DATASET, scan=r.scan,
                                                       path=str(RAW / r.path_in_archive), out_path=str(out)))
    assert status in ("done", "skipped"), f"{r.file_name}: {error}"
    print(status, out.relative_to(BASE))

In [ ]:
# 5. Check: every visit has the four files, all the same size and orientation
import nibabel as nib
for visit in sorted(OUT.iterdir()):
    files = sorted(visit.glob("*.nii.gz"))
    shapes = {nib.load(f).shape for f in files}
    codes = {nib.aff2axcodes(nib.load(f).affine) for f in files}
    print(visit.name, len(files), "files", shapes, codes)
    assert len(files) == 4 and shapes == {(240, 240, 155)} and codes == {("R", "A", "S")}
print("Done. Testers can download the folder", OUT)